In [ ]:
# Word2Vec Word Embeddings using Gensim on a Custom Corpus

In [11]:
!pip install gensim
import gensim
from gensim.models import Word2Vec
import nltk
import re

# Download necessary NLTK data for stopwords (used in c10e16f8)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

## 1. Define a Custom Corpus

Let's start with a simple set of sentences to serve as our custom corpus.

In [2]:
# Sample sentences
corpus = [
    "I love natural language processing with Gensim.",
    "Word embeddings are powerful for understanding text semantics.",
    "Gensim provides an efficient implementation of Word2Vec.",
    "Neural networks are often used for creating word embeddings.",
    "The cat sat on the mat.",
    "The dog ran in the park.",
    "Machine learning is a fascinating field."
]

## 2. Preprocess the Corpus

Before training Word2Vec, we need to preprocess the text. This typically involves:
1.  **Tokenization:** Breaking down sentences into individual words.
2.  **Lowercasing:** Converting all words to lowercase.
3.  **Removing Punctuation:** Cleaning up special characters.
4.  **Removing Stopwords:** Eliminating common words (like 'the', 'is', 'a') that don't add much semantic value.

In [12]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk # Need nltk for downloads here
import re # Re-import re for this cell's self-contained nature

# Ensure NLTK tokenizers are available for word_tokenize
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('tokenizers/punkt_tab') # specifically mentioned in the error
except LookupError:
    nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase the text
    text = text.lower()
    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize the text
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

# Apply preprocessing to the corpus
processed_corpus = [preprocess_text(sentence) for sentence in corpus]

print("Original Corpus:")
for s in corpus:
    print(f"- {s}")

print("\nProcessed Corpus (first 2 sentences):")
for i, p_s in enumerate(processed_corpus[:2]):
    print(f"- {corpus[i]}\n  -> {p_s}")

Original Corpus:
- I love natural language processing with Gensim.
- Word embeddings are powerful for understanding text semantics.
- Gensim provides an efficient implementation of Word2Vec.
- Neural networks are often used for creating word embeddings.
- The cat sat on the mat.
- The dog ran in the park.
- Machine learning is a fascinating field.

Processed Corpus (first 2 sentences):
- I love natural language processing with Gensim.
  -> ['love', 'natural', 'language', 'processing', 'gensim']
- Word embeddings are powerful for understanding text semantics.
  -> ['word', 'embeddings', 'powerful', 'understanding', 'text', 'semantics']


## 3. Train the Word2Vec Model

Now we'll use Gensim's `Word2Vec` model to train word embeddings on our preprocessed corpus. Key parameters include:

*   `sentences`: Our preprocessed list of tokenized sentences.
*   `vector_size`: The dimensionality of the word vectors (e.g., 100).
*   `window`: The maximum distance between the current and predicted word within a sentence.
*   `min_count`: Ignores all words with total frequency lower than this. Useful for filtering out rare words.
*   `sg`: Training algorithm: 1 for skip-gram, 0 for CBOW (Continuous Bag of Words). Skip-gram is generally better for capturing semantic nuances.

In [13]:
from gensim.models import Word2Vec

# Train the Word2Vec model
model = Word2Vec(
    sentences=processed_corpus,
    vector_size=100,  # Dimensionality of the word vectors
    window=5,         # Context window size
    min_count=1,      # Ignore words with frequency less than this
    workers=4,        # Number of CPU cores to use for training
    sg=0              # 0 for CBOW, 1 for Skip-gram
)

# Build the vocabulary (if not done already during initialization)
model.build_vocab(processed_corpus)

# Train the model
model.train(processed_corpus, total_examples=model.corpus_count, epochs=10)

print("Word2Vec model trained successfully!")

Word2Vec model trained successfully!


## 4. Explore the Word Embeddings

Let's see some results from our trained model.

In [14]:
# Find words similar to a given word
word = "gensim"
if word in model.wv:
    similar_words = model.wv.most_similar(word)
    print(f"Words similar to '{word}':")
    for sw, score in similar_words:
        print(f"- {sw}: {score:.4f}")
else:
    print(f"'{word}' not in vocabulary.")

print("\n")

# Get the vector for a word
word_vector = model.wv['gensim']
print(f"Vector for 'gensim' (first 10 dimensions): {word_vector[:10]}")
print(f"Vector shape: {word_vector.shape}")

print("\n")

# Perform a word analogy task (e.g., 'cat' is to 'mat' as 'dog' is to...)
if 'cat' in model.wv and 'mat' in model.wv and 'dog' in model.wv:
    analogy_result = model.wv.most_similar(positive=['dog', 'mat'], negative=['cat'], topn=1)
    print(f"'cat' is to 'mat' as 'dog' is to... {analogy_result[0][0]} (score: {analogy_result[0][1]:.4f})")
else:
    print("Not enough words in vocabulary for analogy task (need 'cat', 'mat', 'dog').")

Words similar to 'gensim':
- implementation: 0.2529
- machine: 0.2009
- language: 0.1702
- efficient: 0.1502
- word: 0.1387
- networks: 0.1085
- sat: 0.0992
- embeddings: 0.0348
- used: 0.0332
- park: 0.0199


Vector for 'gensim' (first 10 dimensions): [-0.00713902  0.00124103 -0.00717672 -0.00224462  0.0037193   0.00583312
  0.00119818  0.00210273 -0.00411039  0.00722533]
Vector shape: (100,)


'cat' is to 'mat' as 'dog' is to... field (score: 0.1962)
